# 11 · STAC discovery and GeoLibre handoff

**Time:** 40 minutes · **Audience:** undergraduate / introductory graduate GIS and Earth science.

**By the end:** Inspect STAC assets, understand CORS and formats, and transfer a documented selection to a browser GIS.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-zarr-sql-views/lite/lab/index.html?path=11_stac_geolibre.ipynb) · [Earth lab](https://jltobias.github.io/JupyterLite-zarr-sql-views/lab/) · [Sources and licenses](https://jltobias.github.io/JupyterLite-zarr-sql-views/book/credits.html)

Run cells from top to bottom with **Shift+Enter**. The first kernel start downloads Python WebAssembly packages.
The bundled exercises need no data-service account. Save a copy with **File → Save and Export Notebook As** before clearing browser storage.

![Four-stage workflow: STAC discovery, Zarr reading, SQL selection, linked views](assets/pipeline.svg)

In [1]:
from course import *
import numpy as np
import matplotlib.pyplot as plt
print('Ready: Python + NumPy + local teaching data')

Ready: Python + NumPy + local teaching data


## 1. Separate metadata from pixels

STAC catalogs describe collections and items; asset links point to data. A COG asset is not a Zarr store.
The observed lesson’s source COGs were prepared outside the browser into a small Zarr extract using a reproducible script.
Catalog access does not guarantee cross-origin permission for every asset.

In [2]:
provenance=json.loads(Path('data/deafrica-provenance.json').read_text())
item=provenance['records'][-1]
print('Search:',item['search'])
print('Raster:',item['frequency_url'])
print('Source CRS:',provenance['source_crs'],'Teaching CRS:',provenance['output_crs'])

Search: https://explorer.digitalearth.africa/stac/search?collections=wofs_ls_summary_annual&bbox=22.70,-20.50,22.71,-20.49&datetime=2022-01-01/2022-12-31&limit=1
Raster: https://deafrica-services.s3.af-south-1.amazonaws.com/wofs_ls_summary_annual/1-0-0/x203/y050/2022--P1Y/wofs_ls_summary_annual_x203y050_2022--P1Y_frequency.tif
Source CRS: EPSG:6933 Teaching CRS: EPSG:4326


## 2. Make an optional live STAC request

Set `LIVE = True` to request a small metadata response. The default uses the recorded provenance so the lesson can finish
when a service is unavailable. The fallback is explicitly labeled and never silently substitutes invented observations.
Pyodide uses browser fetch, which obeys CORS; desktop Python uses urllib.

In [3]:
LIVE=False
if LIVE:
    try:
        if __import__('sys').platform == 'emscripten':
            from pyodide.http import pyfetch
            response=await pyfetch(item['search'])
            if not response.ok: raise RuntimeError(f'HTTP {response.status}')
            stac=await response.json()
        else:
            import urllib.request
            with urllib.request.urlopen(item['search'],timeout=30) as response: stac=json.load(response)
        print('LIVE STAC items:',len(stac['features']))
        print('Asset names:',list(stac['features'][0]['assets']))
    except Exception as exc:
        print('LIVE REQUEST FAILED:',str(exc),'Use the recorded metadata above; no new observation was obtained.')
else:
    print('RECORDED METADATA MODE:',item['item_id'])

RECORDED METADATA MODE: 199183db-4086-526b-b113-7fbfecf19bf7


## 3. Inspect in a STAC browser

Open [DE Africa Explorer](https://explorer.digitalearth.africa/products/wofs_ls_summary_annual),
or paste `https://explorer.digitalearth.africa/stac` into [STAC Browser](https://radiantearth.github.io/stac-browser/).
Record the item ID, collection, date interval, bounding box, CRS, asset media type, nodata, units, and license.
When a viewer cannot load an asset, inspect the network response and format before assuming the dataset is empty.

In [4]:
meta,water=load_cube('ngami')
print('Exported',export_geojson(meta,water,'ngami-for-geolibre.geojson',.5),'sampled points')

Exported 132 sampled points


## 4. Handoff to GeoLibre

1. Download `ngami-for-geolibre.geojson` from the JupyterLite file browser (or use the lab export button).
2. Open [GeoLibre Web](https://web.geolibre.app/) and add the local GeoJSON file using its data import controls.
3. Style points by `value`, retain `time`, and inspect the `kind` field.
4. Compare its 2D or 3D presentation with the course lab. Read GeoLibre’s current tool documentation before using analyses.
5. Save your project with a caption stating that these are sampled locations rather than areal flood polygons.

No GeoLibre Python installation or credential is required for this file-based handoff.
Other useful browser tools include [EO Browser](https://apps.sentinel-hub.com/eo-browser/),
[Copernicus Browser](https://browser.dataspace.copernicus.eu/), and [NASA Worldview](https://worldview.earthdata.nasa.gov/).
Some tools or downloads require their own accounts. Their availability and terms remain external to this course.

## Try it yourself

Inspect one additional DE Africa collection. Explain why its source asset can or cannot be opened directly by the lab’s Zarr reader.

## Check your reasoning

The current lab accepts the bundled schema, not arbitrary STAC items. COGs, compressed Zarr variants, coordinate systems, and authentication each need an explicit adapter.

## Evidence journal

Write four sentences: **what I observed; how I calculated it; what this cannot establish; what evidence I need next**.
For any figure you reuse, retain its units, date or scenario, data origin, transformations, and attribution.

## Sources and credit

- [Digital Earth Africa WOfS specifications and limitations](https://docs.digitalearthafrica.org/en/latest/data_specs/Landsat_WOfS_specs.html), CC BY 4.0 data.
- [STAC specification](https://stacspec.org/en/about/stac-spec/)
- [GeoLibre source and documentation](https://github.com/opengeos/GeoLibre)

Teaching adaptation of [dzole0311/zarr-sql-views](https://github.com/dzole0311/zarr-sql-views), ISC.
Original teaching code and prose: JupyterLite-zarr-sql-views contributors, ISC. Data keep their separate licenses.